# Vector Retriever

In this notebook, you will create a vector retriever using the neo4j-graphrag Python package with Amazon Bedrock.

You will learn how to:
- Use a vector index to retrieve semantically similar chunks
- Combine vector retrieval with an LLM to answer questions
- Inspect the context used by the LLM

---

## Setup

Import the required Python modules and configure the connection.

In [ ]:
import sys
sys.path.insert(0, '..')

from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.generation import GraphRAG

from config import get_neo4j_driver, get_llm, get_embedder

Create and verify the connection to your Neo4j database.

In [ ]:
driver = get_neo4j_driver()
driver.verify_connectivity()
print("Connected to Neo4j successfully!")

## Initialize LLM and Embedder

Set up the Large Language Model (LLM) and the embedding model for GraphRAG workflows.

- **LLM**: Claude via Amazon Bedrock for generating responses
- **Embedder**: Amazon Titan Text Embeddings V2 for semantic search

In [ ]:
llm = get_llm()
embedder = get_embedder()

print(f"LLM: {llm.model_id}")
print(f"Embedder: {embedder.model_id}")
print(f"Embedding dimensions: {embedder.dimensions}")

## Initialize Vector Retriever

Set up the vector-based retriever for semantic search over your Neo4j knowledge graph.

> Vector search enables semantic retrieval of text chunks from your graph.  
> Instead of keyword matching, it finds the most contextually similar passages to your query, even if the wording is different.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

print("Vector retriever initialized!")

The **VectorRetriever** class:
- Connects to the Neo4j database using the provided `driver`
- Uses the `chunkEmbeddings` vector index for efficient semantic retrieval
- The `embedder` generates embeddings for the query
- Returns the `text` property from matching chunks

> **Tip:** You can modify the `return_properties` list to include additional properties from the retrieved nodes.

## Simple Vector Search

Test the vector search by retrieving the top 5 most relevant text chunks for a given query.

In [ ]:
query = "What are the risks that Apple faces?"
result = vector_retriever.search(query_text=query, top_k=5)

print(f"Query: \"{query}\"")
print(f"Number of results returned: {len(result.items)}\n")

for item in result.items:
    score = item.metadata['score']
    content_preview = str(item.content)[:100]
    print(f"Score: {score:.4f}, Content: {content_preview}...")

**How it works:**
1. The query "What are the risks that Apple faces?" is embedded using Amazon Titan
2. `vector_retriever.search()` finds the top 5 matches based on vector similarity
3. Results show:
   - The similarity score (`Score`)
   - A snippet of the retrieved content (`Content`)

> **Tip:** Inspecting the returned results helps verify relevance and can guide adjustments to chunking or embedding strategies.

## GraphRAG Query

The `GraphRAG` class combines retrieval with LLM generation to answer questions using both semantic search and generative reasoning.

In [ ]:
query = "What are the risks that Apple faces?"

rag = GraphRAG(
    llm=llm,
    retriever=vector_retriever
)

response = rag.search(query, retriever_config={"top_k": 5}, return_context=True)

print(f"Query: \"{query}\"")
print(f"Number of chunks retrieved: {len(response.retriever_result.items)}\n")
print("Answer:")
print(response.answer)

**How it works:**
1. The retriever finds the most relevant text chunks from the Neo4j graph
2. Claude uses the retrieved context to generate a natural language answer
3. The response is grounded in your knowledge graph data

## Inspect the Context

With `return_context=True`, you can see exactly what information was sent to the LLM.

In [ ]:
print("=== Retrieved Context ===")
for i, item in enumerate(response.retriever_result.items):
    print(f"\n[{i+1}] Score: {item.metadata['score']:.4f}")
    content_str = str(item.content)
    preview = content_str[:200] + "..." if len(content_str) > 200 else content_str
    print(f"    {preview}")

## Try Different Queries

Experiment with different questions to see how semantic search finds relevant content.

In [ ]:
queries = [
    "What products does Microsoft reference?",
    "What warnings have Nvidia given?",
    "What companies mention AI in their filings?"
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: \"{query}\"")
    print("="*60)
    
    response = rag.search(query, retriever_config={"top_k": 3})
    print(f"\nAnswer:\n{response.answer}")

## Summary

In this notebook, you learned:

1. **VectorRetriever** - How to create a retriever for semantic search
2. **Simple search** - Using `search()` to find similar chunks
3. **GraphRAG** - Combining retrieval with LLM generation
4. **Context inspection** - Viewing what context the LLM receives

The vector retriever is excellent for:
- Semantic similarity queries
- Finding relevant content without exact keyword matches
- General exploratory questions

In the next notebook, you'll learn to enhance retrieval with **graph context** using the VectorCypherRetriever.

---

**Next:** [VectorCypher Retriever](02_vector_cypher_retriever.ipynb)

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")